# Generate the corrected fixed-[110] Al divacancy structures

This reproduces initial structures for the 20260831 protocol: conventional 3×3×3 FCC, a0=3.9545804060131293 Å, fixed **[110] crystal direction**. Production DFTpy parameters are λ=0.9, μ=0.1 and spacing=0.20 Å. Structure generation alone is not a new energy calculation. Distances are initial minimum-image distances under PBC; the third point lies on a half-cell boundary.


In [ ]:
from pathlib import Path
import os
import sys

candidates = ([Path(os.environ['AL_DEFECTS_REPO'])] if os.environ.get('AL_DEFECTS_REPO') else [])
candidates += [Path.cwd(), *Path.cwd().parents]
REPO = next((p.resolve() for p in candidates if (p / 'scripts' / 'divacancy_analysis_checks.py').is_file()), None)
if REPO is None:
    raise FileNotFoundError('Open this notebook from the repository, or set AL_DEFECTS_REPO to the full repository')
sys.path.insert(0, str(REPO / 'scripts'))
from divacancy_analysis_checks import latest_dftpy_root
from ase.io import write
from divacancy_geometry import build_centered_pristine, enumerate_pairs, DFTPY_A0_A


In [ ]:
pristine, center_index, _ = build_centered_pristine(DFTPY_A0_A, (3, 3, 3))
pairs = enumerate_pairs(pristine, center_index, selection='fixed_direction', direction=(1, 1, 0))
assert len(pairs) == 3, f'Unexpected fixed-[110] structure count: {len(pairs)}'
print('Pristine atoms:', len(pristine), 'cell lengths (Å):', pristine.cell.lengths())
for index, (distance, second_index, delta) in enumerate(pairs, 1):
    print(index, f'r={distance:.8f} Å', 'minimum-image vector=', delta)


In [ ]:
# Save only into a new demo directory; existing datasets are preserved.
from datetime import datetime
base = Path(os.environ.get('AL_DEFECTS_NOTEBOOK_OUTPUT', str(REPO / 'work' / 'notebook_outputs')))
output = base / ('divacancy_structure_demo_' + datetime.now().strftime('%Y%m%d_%H%M%S_%f'))
output.mkdir(parents=True, exist_ok=False)
write(output / 'pristine_start.vasp', pristine, direct=True, vasp5=True)
for index, (distance, second_index, _) in enumerate(pairs, 1):
    defect = pristine.copy()
    del defect[sorted((center_index, second_index), reverse=True)]
    assert len(defect) == 106
    write(output / f'divacancy_pair_{index:02d}.vasp', defect, direct=True, vasp5=True)
print('Wrote new demo:', output)
